In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 59 — Evals-First Fine-Tuning Pipeline

## Goal
Build a **full dataset → eval → tune → compare loop** that treats
evaluation as the *starting point*, not an afterthought.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Evals first** | Write tests before training, like TDD for ML |
| **Baseline measurement** | Know how bad the model is before tuning |
| **Automated scoring** | Build reusable evaluation functions |
| **Before/after comparison** | Quantify improvement from fine-tuning |

## Pipeline
```
1. Define Task & Eval Criteria
2. Build Eval Harness
3. Measure Baseline (zero-shot)
4. Create Training Data
5. Fine-Tune
6. Re-run Evals
7. Compare & Decide
```

## Task: Medical Triage Classification
Classify patient descriptions into urgency levels: `EMERGENCY`, `URGENT`, `ROUTINE`.

In [ ]:
import os, warnings, json, time
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import torch
print(f"PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}")

## Step 1 — Define the Eval Set (Before Training!)

**Key principle**: Write the test set *first*. Never contaminate
eval data with training data.

In [ ]:
eval_set = [
    # EMERGENCY
    {"text": "Patient is unconscious, not breathing, found collapsed on floor.", "label": "EMERGENCY"},
    {"text": "Severe chest pain radiating to left arm, profuse sweating, onset 10 min ago.", "label": "EMERGENCY"},
    {"text": "Child ingested unknown household chemicals 5 minutes ago, vomiting.", "label": "EMERGENCY"},
    {"text": "Major bleeding from deep laceration on thigh, bleeding not stopping with pressure.", "label": "EMERGENCY"},
    # URGENT
    {"text": "High fever of 103°F for 3 days, difficulty swallowing, no improvement with OTC meds.", "label": "URGENT"},
    {"text": "Twisted ankle while hiking, significant swelling, can bear some weight.", "label": "URGENT"},
    {"text": "Persistent abdominal pain for 48 hours, nausea, no appetite.", "label": "URGENT"},
    {"text": "Deep cut on hand from kitchen knife, bleeding controlled but may need stitches.", "label": "URGENT"},
    # ROUTINE
    {"text": "Annual checkup, no current complaints, last visit 11 months ago.", "label": "ROUTINE"},
    {"text": "Mild seasonal allergies, runny nose and sneezing for a week.", "label": "ROUTINE"},
    {"text": "Request for prescription refill, blood pressure medication, stable for 2 years.", "label": "ROUTINE"},
    {"text": "Slight knee stiffness after exercise, no swelling, manageable discomfort.", "label": "ROUTINE"},
]

print(f"Eval set: {len(eval_set)} examples")
from collections import Counter
label_counts = Counter(e["label"] for e in eval_set)
print(f"Distribution: {dict(label_counts)}")

## Step 2 — Build the Eval Harness

Write reusable scoring functions **before** any training.
This ensures we measure the same thing before and after.

In [ ]:
VALID_LABELS = {"EMERGENCY", "URGENT", "ROUTINE"}

def extract_label(response: str) -> str:
    """Extract the triage label from free-text model output."""
    response_upper = response.upper()
    for label in VALID_LABELS:
        if label in response_upper:
            return label
    return "UNKNOWN"

def evaluate_model(predict_fn, eval_data, model_name="model"):
    """
    Run eval harness on a predict function.
    Returns dict with accuracy, per-class accuracy, and details.
    """
    results = []
    correct = 0
    class_correct = Counter()
    class_total = Counter()

    for ex in eval_data:
        raw = predict_fn(ex["text"])
        pred = extract_label(raw)
        is_correct = pred == ex["label"]
        correct += int(is_correct)
        class_total[ex["label"]] += 1
        class_correct[ex["label"]] += int(is_correct)
        results.append({
            "text": ex["text"][:60],
            "expected": ex["label"],
            "predicted": pred,
            "correct": is_correct,
        })

    accuracy = correct / len(eval_data)
    per_class = {c: class_correct[c] / class_total[c] for c in sorted(VALID_LABELS)}

    print(f"\n{'='*50}")
    print(f" {model_name} — Eval Results")
    print(f"{'='*50}")
    print(f" Overall Accuracy: {accuracy:.1%} ({correct}/{len(eval_data)})")
    for cls, acc in per_class.items():
        print(f"   {cls:12s}: {acc:.1%} ({class_correct[cls]}/{class_total[cls]})")
    print()
    for r in results:
        icon = "✅" if r["correct"] else "❌"
        print(f"  {icon} {r['expected']:10s} → {r['predicted']:10s}  {r['text']}")

    return {"accuracy": accuracy, "per_class": per_class, "details": results}

print("✅ Eval harness defined")

## Step 3 — Measure Baseline (Zero-Shot)

Before any training, see how the base model performs.
This tells us **how much room there is for improvement**.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")

TRIAGE_PROMPT = """Classify the urgency of this patient description.
Reply with EXACTLY one word: EMERGENCY, URGENT, or ROUTINE.

Patient: {text}

Classification:"""

def predict_baseline(text: str) -> str:
    messages = [{"role": "user", "content": TRIAGE_PROMPT.format(text=text)}]
    t = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(t, return_tensors="pt").to(base_model.device)
    with torch.no_grad():
        out = base_model.generate(**inputs, max_new_tokens=20, temperature=0.1, do_sample=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

baseline_results = evaluate_model(predict_baseline, eval_set, "Baseline (zero-shot)")

## Step 4 — Create Training Data

Now we know the baseline. Build training data targeting the
failure modes we observed. **Do not include eval examples.**

In [ ]:
SYS_MSG = "You are a medical triage classifier. Given a patient description, classify as EMERGENCY, URGENT, or ROUTINE. Reply with exactly one word."

train_examples = [
    # EMERGENCY
    ("Cardiac arrest, CPR in progress by bystander, AED requested.", "EMERGENCY"),
    ("Severe anaphylaxis after bee sting, throat closing, patient carrying EpiPen.", "EMERGENCY"),
    ("Stroke symptoms: sudden facial drooping, slurred speech, right arm weakness.", "EMERGENCY"),
    ("Motorcycle accident victim, visible bone fracture, heavy bleeding.", "EMERGENCY"),
    ("Patient found unresponsive, suspected opioid overdose, shallow breathing.", "EMERGENCY"),
    ("Infant not breathing, turning blue, parents panicking.", "EMERGENCY"),
    ("Severe burns covering both arms from grease fire, blistering.", "EMERGENCY"),
    ("Gunshot wound to abdomen, conscious but in severe distress.", "EMERGENCY"),
    # URGENT
    ("Moderate asthma attack, rescue inhaler used twice with partial relief.", "URGENT"),
    ("Suspected UTI, painful urination for 3 days, mild fever.", "URGENT"),
    ("Dog bite on forearm, puncture wound, last tetanus shot unknown.", "URGENT"),
    ("Migraine lasting 2 days with visual disturbances, no relief from medication.", "URGENT"),
    ("Toddler with ear infection symptoms, pulling ear, fussy, low-grade fever.", "URGENT"),
    ("Possible broken wrist after fall, swelling and bruising, can move fingers.", "URGENT"),
    ("Persistent vomiting for 24 hours, unable to keep fluids down.", "URGENT"),
    ("Second-degree sunburn covering back and shoulders, blistering.", "URGENT"),
    # ROUTINE
    ("Follow-up visit for well-controlled Type 2 diabetes, A1c check due.", "ROUTINE"),
    ("New patient registration, needs general physical exam.", "ROUTINE"),
    ("Mild back pain from sitting at desk, intermittent, two weeks.", "ROUTINE"),
    ("Vaccination appointment for flu shot, no current illness.", "ROUTINE"),
    ("Routine eye exam referral, last exam 18 months ago.", "ROUTINE"),
    ("Wants mole checked, unchanged for years, no pain or itching.", "ROUTINE"),
    ("Annual well-child visit, developmental milestones on track.", "ROUTINE"),
    ("Insomnia improving with sleep hygiene changes, follow-up discussion.", "ROUTINE"),
]

train_data = [{"messages": [
    {"role": "system", "content": SYS_MSG},
    {"role": "user", "content": f"Patient: {text}"},
    {"role": "assistant", "content": label},
]} for text, label in train_examples]

print(f"Training examples: {len(train_data)}")
print(f"Distribution: {Counter(ex[1] for ex in train_examples)}")

## Step 5 — Fine-Tune

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# Free base model memory
del base_model
torch.cuda.empty_cache()

OUTPUT_DIR = "./triage_finetuned"

trainer = SFTTrainer(
    model=MODEL_ID,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        max_steps=80,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=5,
        max_length=256,
        bf16=True,
        logging_steps=10,
        gradient_checkpointing=True,
        report_to="none",
    ),
    train_dataset=Dataset.from_list(train_data),
    peft_config=LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

print("Fine-tuning on triage data...")
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f"✅ Training done! Loss: {train_result.training_loss:.4f}")

## Step 6 — Re-run Evals on Fine-Tuned Model

In [ ]:
from peft import AutoPeftModelForCausalLM

ft_model = AutoPeftModelForCausalLM.from_pretrained(OUTPUT_DIR, torch_dtype=torch.bfloat16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def predict_finetuned(text: str) -> str:
    messages = [
        {"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Patient: {text}"},
    ]
    t = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(t, return_tensors="pt").to(ft_model.device)
    with torch.no_grad():
        out = ft_model.generate(**inputs, max_new_tokens=10, temperature=0.1, do_sample=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

ft_results = evaluate_model(predict_finetuned, eval_set, "Fine-Tuned")

## Step 7 — Side-by-Side Comparison

In [ ]:
import matplotlib.pyplot as plt

labels_list = sorted(VALID_LABELS)
base_acc = [baseline_results["per_class"].get(c, 0) for c in labels_list]
ft_acc = [ft_results["per_class"].get(c, 0) for c in labels_list]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Per-class accuracy
x = range(len(labels_list))
axes[0].bar([i - 0.18 for i in x], base_acc, width=0.35, label="Baseline", color="salmon")
axes[0].bar([i + 0.18 for i in x], ft_acc, width=0.35, label="Fine-Tuned", color="mediumseagreen")
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(labels_list)
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Per-Class Accuracy")
axes[0].set_ylim(0, 1.15)
axes[0].legend()

# Overall accuracy
axes[1].bar(["Baseline", "Fine-Tuned"],
            [baseline_results["accuracy"], ft_results["accuracy"]],
            color=["salmon", "mediumseagreen"])
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Overall Accuracy")
axes[1].set_ylim(0, 1.15)
for i, v in enumerate([baseline_results["accuracy"], ft_results["accuracy"]]):
    axes[1].text(i, v + 0.03, f"{v:.1%}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

# Summary table
print(f"\n{'Metric':<20} {'Baseline':<12} {'Fine-Tuned':<12} {'Δ':<10}")
print("─" * 55)
delta = ft_results['accuracy'] - baseline_results['accuracy']
print(f"{'Overall Accuracy':<20} {baseline_results['accuracy']:<12.1%} {ft_results['accuracy']:<12.1%} {delta:+.1%}")
for cls in labels_list:
    b = baseline_results['per_class'].get(cls, 0)
    f = ft_results['per_class'].get(cls, 0)
    print(f"{cls:<20} {b:<12.1%} {f:<12.1%} {f-b:+.1%}")

## 🧠 Key Concepts Recap

| Step | What | Why |
|------|------|-----|
| 1. Eval set first | Write test cases before training | Prevents data leakage, sets success criteria |
| 2. Eval harness | Reusable scoring functions | Same measurement before & after |
| 3. Baseline | Measure zero-shot performance | Know room for improvement |
| 4. Train data | Separate from eval, targeting weaknesses | Honest evaluation |
| 5. Fine-tune | SFT with LoRA | Efficient training |
| 6. Re-eval | Same harness, same data | Apples-to-apples comparison |
| 7. Decide | Compare metrics, visualize | Data-driven go/no-go |

## Evals-First Mindset
```
Traditional:  Data → Train → "Looks good" → Ship
Evals-First:  Define success → Measure baseline → Train → Prove improvement → Ship
```

## When to Fine-Tune vs. Not
- **Fine-tune**: Baseline is poor, you have domain data, task is narrow
- **Don't tune**: Baseline already good, prompt engineering suffices, data is scarce